# `NSE` json tests

In [ ]:
# hack to change jupyter directory in notebooks for imports
import os
from pathlib import Path
if Path.cwd().parts[-1] == 'notebooks':
    root = Path.cwd().parent
else:
    root = Path.cwd()
os.chdir(root)

# Logger
import logging, logging.config
import yaml

with open(root / 'config' / 'log.yml', 'r') as f:
    config = yaml.safe_load(f.read())
    logging.config.dictConfig(config)

log = logging.getLogger('ib_log')

## Reading json files from NSE website

In [ ]:
# Function in src/nse/nse_json()

# ... to read json files

import requests
import pandas as pd
import numpy as np

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:74.0) Gecko/20100101 Firefox/74.0'
}

def nse_json(url: str):
    """Fetch json from nse for the url provided"""
    try:
        output = requests.get(url,headers=headers).json()

    except ValueError:
        try:
            s=requests.Session()
            output = s.get("http://nseindia.com",headers=headers)
            output = s.get(url,headers=headers).json()

        except ValueError: # for csv loads generating JSONDecodeError
            output = requests.get(url).text
        
    return output

In [ ]:
# from src.nse.nse import nse_json

import pandas as pd
import numpy as np

symbol = "INFY"

## Getting ALL equity prices

In [ ]:
# Function in nse/nse/equity_prices()

def equity_prices() -> pd.DataFrame:
    """Gets all live equity prices"""

    url = "https://www1.nseindia.com/live_market/dynaContent/live_watch/stock_watch/foSecStockWatch.json"

    js = nse_json(url)
    x = []

    for item in js['data']:
        df = pd.DataFrame.from_dict([item])
        x.append(df)
    df = pd.concat(x, ignore_index=True)

    return df

In [6]:
# testing equity price watch url

url = "https://www1.nseindia.com/live_market/dynaContent/live_watch/stock_watch/foSecStockWatch.json"

js = nse_json(url)
x = []

for item in js['data']:
    df = pd.DataFrame.from_dict([item])
    x.append(df)
df = pd.concat(x, ignore_index=True)

df.head()

## Getting ALL Index prices

In [ ]:
# testing an index price watch url

url = 'https://iislliveblob.niftyindices.com/jsonfiles/LiveIndicesWatch.json'
js = nse_json(url) # works!

x = []
for item in js['data']:
    df = pd.DataFrame.from_dict([item])
    x.append(df)
df = pd.concat(x, ignore_index=True)

# keep only equities
df = df[(df.indexType == 'eq') & (df.yearHigh != '-')].reset_index(drop=True)

# symbol map
di = {'NIFTY 50': 'NIFTY50', 'NIFTY BANK': 'BANKNIFTY', 
      'INDIA VIX': 'INDIAVIX', 'NIFTY IT': 'NIFTYIT'}
df.insert(0, 'symbol', df.indexName.map(di).fillna(df.indexName))

important = True

# keep only important indices
if important:
   df = df[df.symbol.isin(di.values())]


In [ ]:
df

## Checking if NSE is open now

In [ ]:
# testing a plan marketStatus url
url = 'https://nseindia.com/api/marketStatus'
js = nse_json(url) # works!
nse_is_open = js[list(js.keys())[0]][0]['marketStatus'] != 'Closed'
nse_is_open

## Parsing a csv file from NSE website
... can be used to liberate get_nse_syms!

In [ ]:
# testing a url with a csv file
# similar to src/nse/nse.py/get_nse_syms()

symbol = "" # Empty means all symbols

url = "https://archives.nseindia.com/content/fo/fo_mktlots.csv"
js = nse_json(url)

df = pd.read_csv(url)
df.columns = df.columns.str.strip().str.lower() # cleanup column names

# strip all string contents of whitespaces
df = df.applymap(lambda x: x.strip()
         if type(x) is str else x)

# remove 'Symbol' row
df = df[df.symbol != "Symbol"]

# introduce `secType`
df.insert(1, 'secType', np.where(df.symbol.str.contains('NIFTY'), 'IND', 'STK'))

# introduce `exchange`
df.insert(2, 'exchange', 'NSE')

if symbol:
    result = df[(df.iloc[:, 1] == symbol.upper())]
else:
    result = df


# Testing to liberate `history`    
...from nsepy and other dependancies


In [ ]:
# testing equity history
from functools import partial
from urllib.parse import urlparse


In [ ]:
url1 = "http://www1.nseindia.com/products/dynaContent/common/productsSymbolMapping.jsp"
partial(urlparse(url1))

In [ ]:


url2 = "symbol=SBIN&segmentLink=3&symbolCount=1&series=EQ&dateRange=1month&fromDate='01-01-2017'&toDate='01-01-2017'&dataType=PRICEVOLUMEDELIVERABLE"
partial(url1, url2)

In [ ]:

import requests

class URLFetch:

    def __init__(self, url, method='get', json=False, session=None,
                 headers = None, proxy = None):
        self.url = url
        self.method = method
        self.json = json

        if not session:
            self.session = requests.Session()
        else:
            self.session = session

        if headers:
            self.session.headers.update(headers)
        if proxy:
            self.update_proxy(proxy)
        else:
            self.update_proxy('')

    def set_session(self, session):
        self.session = session
        return self

    def get_session(self, session):
        self.session = session
        return self

    def __call__(self, *args, **kwargs):
        u = urlparse(self.url)
        self.session.headers.update({'Host': u.hostname})
        url = self.url%(args)
        if self.method == 'get':
            return self.session.get(url, params=kwargs, proxies = self.proxy )
        elif self.method == 'post':
            if self.json:
                return self.session.post(url, json=kwargs, proxies = self.proxy )
            else:
                return self.session.post(url, data=kwargs, proxies = self.proxy )

In [ ]:
eq_hist_partial = "http://www1.nseindia.com/products/dynaContent/common/productsSymbolMapping.jsp"

symbol="SBIN"
symbolCount=1
series="EQ"
fromDate="dd-mm-yyyy"
toDate="dd-mm-yyyy"
dd = "symbol='SBIN', series='EQ', fromDate='01-01-2017', toDate='01-01-2017'"